#Init 

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

#Silver transformations

##Trimming 

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, T.StringType):
        trimmed_col = F.trim(F.col(field.name))
        
        df = df.withColumn(
            field.name, 
            F.when(trimmed_col == "", F.lit(None).cast("string"))
             .otherwise(trimmed_col)
        )

##Customer ID cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(F.col("cid"), "-", ""))

##Standarization

In [0]:
COUNTRY_STANDARDIZATION = {
    "AUSTRALIA": "Australia",
    "UNITED STATES": "United States", 
    "US": "United States",
    "USA": "United States",
    "UNITED KINGDOM": "United Kingdom",
    "UK": "United Kingdom",
    "FRANCE": "France",
    "CANADA": "Canada",
    "GERMANY": "Germany",
    "DE": "Germany"
}

In [0]:
map_elements = []
for key, value in COUNTRY_STANDARDIZATION.items():
    map_elements.extend([F.lit(key), F.lit(value)])
    
spark_map = F.create_map(*map_elements)
df = df.withColumn(
    "CNTRY",
    F.coalesce(
        spark_map[F.upper(F.col("CNTRY"))],
        F.lit(None).cast("string") 
    )
)

##Renaming Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "CNTRY": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks Dataframe

In [0]:
df.limit(10).display()

#Writing in Silver Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.silver.erp_customer_location")

##Sanity Checks Table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10